# Transformers Pipeline

The `pipeline()` function is the highest-level API in HuggingFace Transformers. It handles **tokenization → model inference → post-processing** in a single call, letting you run state-of-the-art models without knowing the internals.

In this notebook we explore 6 different NLP tasks, understand batching for efficiency, and peek inside how a pipeline works.

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers pillow -q

## Imports

In [ ]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
print(f"PyTorch {torch.__version__} | device: {'GPU' if device == 0 else 'CPU'}")

## 1. Sentiment Analysis

Classify text as positive or negative. The default model is `distilbert-base-uncased-finetuned-sst-2-english`.

In [ ]:
sentiment = pipeline("sentiment-analysis", device=device)

texts = [
    "I absolutely love working with Hugging Face models!",
    "This API is confusing and the documentation is terrible.",
    "The weather today is neither good nor bad.",
]

for text, result in zip(texts, sentiment(texts)):
    label = result["label"]
    score = result["score"]
    print(f"[{label:8s} {score:.2f}]  {text}")

## 2. Named Entity Recognition (NER)

NER identifies and classifies named entities (persons, organizations, locations, dates) in text.

In [ ]:
ner = pipeline("ner", aggregation_strategy="simple", device=device)

text = "Elon Musk founded SpaceX in 2002 in Hawthorne, California."
entities = ner(text)

print(f"Text: {text}\n")
for ent in entities:
    print(f"  {ent['entity_group']:5s}  '{ent['word']}'  (score: {ent['score']:.2f})")

## 3. Question Answering (Extractive)

Given a **context** paragraph and a **question**, the model extracts the answer span directly from the context.

> **Note:** The `"question-answering"` pipeline shortcut was removed in Transformers 5.x. We use `AutoModelForQuestionAnswering` directly — which is actually more instructive, as it shows the span-extraction mechanics under the hood.

In [ ]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
import torch

model_name = "distilbert-base-cased-distilled-squad"
qa_tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(model_name)

def answer_question(question, context):
    inputs = qa_tokenizer(question, context, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = qa_model(**inputs)
    start = outputs.start_logits.argmax()
    end   = outputs.end_logits.argmax() + 1
    return qa_tokenizer.decode(inputs["input_ids"][0][start:end])

context = """
Hugging Face is a machine learning company based in New York City and Paris.
It was founded in 2016 and is best known for its Transformers library,
which provides thousands of pre-trained models for NLP, vision, and audio tasks.
"""

questions = [
    "Where is Hugging Face based?",
    "When was Hugging Face founded?",
    "What is Hugging Face best known for?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {answer_question(q, context)}\n")

## 4. Fill-Mask

BERT-style models are trained to predict a **masked token** using bidirectional context — both what comes before and after the `[MASK]`. This is useful for understanding how models represent language.

> **Note:** The `"summarization"` pipeline was removed in Transformers 5.x. Fill-mask demonstrates a complementary skill: bidirectional language understanding (vs. the autoregressive generation shown in Task 5).

In [ ]:
unmasker = pipeline("fill-mask", model="bert-base-uncased", device=device)

sentences = [
    "The capital of France is [MASK].",
    "The [MASK] library is the most popular NLP toolkit.",
    "Neural networks learn [MASK] from data.",
]

print("Fill-Mask predictions (top 3):\n")
for sentence in sentences:
    results = unmasker(sentence, top_k=3)
    print(f"Input: {sentence}")
    for r in results:
        print(f"  → '{r['token_str']:15s}'  score: {r['score']:.3f}")
    print()

## 5. Text Generation

Autoregressive models like GPT-2 predict the **next token** given all previous tokens. This is the foundation of modern LLMs.

> **Note:** The `"translation"` pipeline was removed in Transformers 5.x. Text generation is the more relevant paradigm today — instruction-tuned LLMs handle translation as a special case of generation.

In [ ]:
generator = pipeline("text-generation", model="gpt2", device=device)

prompts = [
    "The future of artificial intelligence is",
    "Open source models are important because",
    "The best way to learn machine learning is",
]

print("Text Generation with GPT-2:\n")
for prompt in prompts:
    result = generator(prompt, max_new_tokens=25, do_sample=True, temperature=0.8, top_p=0.9)[0]
    print(f"Prompt : {prompt}")
    print(f"Output : {result['generated_text']}")
    print()

## 6. Zero-Shot Classification

Classify text into **any categories you define** — no task-specific training needed. The model reasons about whether the text entails each label.

In [ ]:
zero_shot = pipeline("zero-shot-classification", device=device)

texts_and_labels = [
    ("The new iPhone 15 features a titanium chassis and USB-C connector.", ["technology", "sports", "politics", "finance"]),
    ("The central bank raised interest rates by 25 basis points today.", ["technology", "sports", "politics", "finance"]),
    ("The team won the championship after a dramatic penalty shootout.", ["technology", "sports", "politics", "finance"]),
]

for text, labels in texts_and_labels:
    result = zero_shot(text, candidate_labels=labels)
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    print(f"Text  : {text[:60]}...")
    print(f"Label : {top_label} ({top_score:.2f})\n")

## How Pipelines Work Internally

Every `pipeline()` call does three things under the hood:

```
Raw text
   │
   ▼
[Tokenizer]  → converts text to token IDs (integers)
   │
   ▼
[Model]      → runs transformer forward pass, returns logits/embeddings
   │
   ▼
[Post-processor] → converts model output to human-readable result
```

You can access each component directly:

In [ ]:
# Peek inside a pipeline
pipe = pipeline("sentiment-analysis", device=device)

tokenizer = pipe.tokenizer
model     = pipe.model

text = "Transformers are incredibly powerful!"

# Step 1: tokenize
tokens = tokenizer(text, return_tensors="pt")
print("Token IDs :", tokens["input_ids"])
print("Tokens    :", tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))

# Step 2: model forward pass
import torch
with torch.no_grad():
    outputs = model(**tokens)

print("\nRaw logits:", outputs.logits)
probs = torch.softmax(outputs.logits, dim=-1)
labels = model.config.id2label
print("Probabilities:", {labels[i]: f"{p:.3f}" for i, p in enumerate(probs[0].tolist())})